# Local AI Code Assistant
### Powered by Qwen 2.5 (local) + Optional Gemini API

A local-first AI coding assistant that:
- Runs **Qwen 2.5** locally (via Ollama) for code generation, explanation, and debugging
- Optionally calls **Gemini API** for tasks that benefit from a larger cloud model
- Provides an interactive **planner UI** — describe a feature, break it into steps, generate code, and track progress

---

## 1. Setup & Install Dependencies

In [20]:
# Install required packages
!pip install -q requests google-generativeai python-dotenv ipywidgets

## 2. Configuration (from `.env`)

Loads settings from `.env` file. Edit `.env` to configure:
- `OLLAMA_HOST` — Ollama server URL
- `QWEN_MODEL` — which Qwen model to use
- `GEMINI_API_KEY` — optional Gemini API key
- `DEFAULT_ENGINE` — `local` or `gemini`

In [21]:
import os
from dotenv import load_dotenv

load_dotenv()

# --- Configuration (from plan section 4) ---
OLLAMA_HOST = os.getenv('OLLAMA_HOST', 'http://localhost:11434')
QWEN_MODEL = os.getenv('QWEN_MODEL', 'qwen2.5:7b')
GEMINI_API_KEY = os.getenv('GEMINI_API_KEY')
GEMINI_MODEL = os.getenv('GEMINI_MODEL', 'gemini-1.5-flash')
DEFAULT_ENGINE = os.getenv('DEFAULT_ENGINE', 'local')
APP_TITLE = os.getenv('APP_TITLE', 'AI Code Planner')

print('Config loaded')
print('  Ollama Host :', OLLAMA_HOST)
print('  Qwen Model  :', QWEN_MODEL)
gemini_ok = GEMINI_API_KEY and GEMINI_API_KEY != 'your_gemini_api_key_here'
print('  Gemini Key  :', 'Set' if gemini_ok else 'Not set')
print('  Default Engine:', DEFAULT_ENGINE)

Config loaded
  Ollama Host : http://localhost:11434
  Qwen Model  : qwen2.5:7b
  Gemini Key  : Not set
  Default Engine: local


## 3. Ollama Health Check

Verify Ollama is running and the Qwen model is available.

In [22]:
import requests

def check_ollama_health():
    """Check if Ollama is running and the Qwen model is available."""
    try:
        resp = requests.get(OLLAMA_HOST + '/api/tags', timeout=5)
        resp.raise_for_status()
        models = resp.json().get('models', [])
        model_names = [m['name'] for m in models]
        print('Ollama is running at ' + OLLAMA_HOST)
        print('  Available models: ' + str(model_names))
        if any(QWEN_MODEL in name for name in model_names):
            print('  Model ' + QWEN_MODEL + ' is available')
        else:
            print('  WARNING: Model ' + QWEN_MODEL + ' not found. Run: ollama pull ' + QWEN_MODEL)
        return True
    except requests.ConnectionError:
        print('ERROR: Cannot connect to Ollama at ' + OLLAMA_HOST)
        print('  Start it with: ollama serve')
        return False
    except Exception as e:
        print('ERROR checking Ollama: ' + str(e))
        return False

ollama_available = check_ollama_health()

Ollama is running at http://localhost:11434
  Available models: ['qwen2.5:7b']
  Model qwen2.5:7b is available


## 4. Prompt Templates (from plan section 5.4 — `prompts.py`)

All prompt templates used by the planner and code generator.

In [5]:
# --- Prompt Templates (plan section 5.4) ---

TASK_BREAKER_SYSTEM = """You are an expert software architect and project planner.
When given a feature description, break it down into clear, ordered implementation steps.

Rules:
- Return ONLY a numbered list of tasks (1. 2. 3. etc.)
- Each task should be a single, concrete implementation step
- Order tasks by dependency (foundational work first)
- Keep each task concise but specific
- Aim for 4-8 tasks for a typical feature
- Do NOT include explanations outside the numbered list"""

TASK_BREAKER_USER = """Break down this feature into implementation steps:

{feature_description}"""


CODE_GENERATOR_SYSTEM = """You are an expert software developer.
Generate clean, well-commented, production-ready code for the given task.

Rules:
- Write complete, runnable code (not pseudocode)
- Include necessary imports
- Add clear comments explaining key logic
- Follow best practices for the language/framework
- After the code block, provide a brief explanation (2-3 sentences)"""

CODE_GENERATOR_USER = """Generate code for this task:

Task: {task_description}

Context (overall project): {project_description}"""


CODE_EXPLAIN_SYSTEM = """You are an expert code reviewer.
Explain the given code clearly and concisely.

Rules:
- Explain what the code does at a high level first
- Then walk through key sections
- Mention any potential issues or improvements
- Keep it concise but thorough"""

CODE_EXPLAIN_USER = """Explain this code:

```
{code}
```"""


DEBUG_SYSTEM = """You are an expert debugger.
Help identify and fix bugs in the given code.

Rules:
- Identify the likely bug(s)
- Explain why it is a bug
- Provide the corrected code
- Suggest how to prevent similar bugs"""

DEBUG_USER = """Debug this code:

```
{code}
```

Error/Issue: {error_description}"""

print('Prompt templates loaded')

Prompt templates loaded


## 5. Qwen 2.5 Client — Local LLM via Ollama (plan section 5.1)

Wraps Ollama REST API (`/api/chat`) for local Qwen 2.5 inference.  
No API key needed — fully offline once the model is pulled.

In [23]:
import requests
import json

class QwenClient:
    """Wraps Ollama/Qwen 2.5 calls (plan section 5.1 — qwen_client.py)."""

    def __init__(self, host=None, model=None):
        self.host = host or OLLAMA_HOST
        self.model = model or QWEN_MODEL
        self.chat_url = self.host + '/api/chat'

    def generate(self, prompt, system_prompt='', temperature=0.7, max_tokens=2048):
        messages = []
        if system_prompt:
            messages.append({'role': 'system', 'content': system_prompt})
        messages.append({'role': 'user', 'content': prompt})
        payload = {
            'model': self.model,
            'messages': messages,
            'stream': False,
            'options': {
                'temperature': temperature,
                'num_predict': max_tokens
            }
        }
        try:
            resp = requests.post(self.chat_url, json=payload, timeout=120)
            resp.raise_for_status()
            data = resp.json()
            return data['message']['content']
        except requests.ConnectionError:
            return 'ERROR: Cannot connect to Ollama. Is it running? (ollama serve)'
        except requests.Timeout:
            return 'ERROR: Request timed out. The model may be loading.'
        except Exception as e:
            return 'ERROR: ' + str(e)

    def is_available(self):
        try:
            resp = requests.get(self.host + '/api/tags', timeout=5)
            resp.raise_for_status()
            models = [m['name'] for m in resp.json().get('models', [])]
            return any(self.model in name for name in models)
        except:
            return False

qwen_client = QwenClient()
print('QwenClient initialized (model: ' + QWEN_MODEL + ')')
print('  Available: ' + str(qwen_client.is_available()))

QwenClient initialized (model: qwen2.5:7b)
  Available: True


## 6. Gemini Client — Optional Cloud Fallback (plan section 5.2)

Used only when `DEFAULT_ENGINE=gemini` or toggled in the UI.  
Reads `GEMINI_API_KEY` from `.env`.

In [24]:
class GeminiClient:
    """Wraps Gemini API calls (plan section 5.2 — gemini_client.py)."""

    def __init__(self, api_key=None, model=None):
        self.api_key = api_key or GEMINI_API_KEY
        self.model_name = model or GEMINI_MODEL
        self._model = None
        if self.is_available():
            self._init_client()

    def _init_client(self):
        try:
            import google.generativeai as genai
            genai.configure(api_key=self.api_key)
            self._model = genai.GenerativeModel(self.model_name)
        except Exception as e:
            print('Gemini init error: ' + str(e))
            self._model = None

    def generate(self, prompt, system_prompt='', temperature=0.7, max_tokens=2048):
        if not self._model:
            return 'ERROR: Gemini not configured. Set GEMINI_API_KEY in .env'
        try:
            full_prompt = system_prompt + '\n\n' + prompt if system_prompt else prompt
            config = {'temperature': temperature, 'max_output_tokens': max_tokens}
            response = self._model.generate_content(full_prompt, generation_config=config)
            return response.text
        except Exception as e:
            return 'Gemini Error: ' + str(e)

    def is_available(self):
        return bool(self.api_key and self.api_key != 'your_gemini_api_key_here')

gemini_client = GeminiClient()
print('GeminiClient initialized')
print('  Available: ' + str(gemini_client.is_available()))

GeminiClient initialized
  Available: False


## 7. Router — Model Dispatcher (plan section 5.3)

Single interface `generate(prompt, engine)` that dispatches to Qwen or Gemini transparently.

In [17]:
class Router:
    """Picks local vs cloud model (plan section 5.3 — router.py)."""

    def __init__(self, qwen_client, gemini_client, default_engine=None):
        self.qwen = qwen_client
        self.gemini = gemini_client
        self.default_engine = default_engine or DEFAULT_ENGINE

    def generate(self, prompt, system_prompt='', engine=None, temperature=0.7, max_tokens=2048):
        engine = engine or self.default_engine
        if engine == 'gemini' and self.gemini.is_available():
            return self.gemini.generate(prompt, system_prompt, temperature, max_tokens)
        elif engine == 'gemini' and not self.gemini.is_available():
            print('Gemini not available, falling back to local Qwen')
            return self.qwen.generate(prompt, system_prompt, temperature, max_tokens)
        else:
            return self.qwen.generate(prompt, system_prompt, temperature, max_tokens)

    def get_available_engines(self):
        engines = []
        if self.qwen.is_available():
            engines.append('local')
        if self.gemini.is_available():
            engines.append('gemini')
        return engines if engines else ['local']

router = Router(qwen_client, gemini_client)
print('Router initialized')
print('  Default engine: ' + router.default_engine)
print('  Available engines: ' + str(router.get_available_engines()))

Router initialized
  Default engine: local
  Available engines: ['local']


## 8. Task Breaker (plan section 5.4 — `task_breaker.py`)

Takes a feature description and returns an ordered list of subtasks.

In [18]:
import re

class TaskBreaker:
    """Splits a feature request into steps with error handling & fallback parsing."""

    def __init__(self, router):
        self.router = router

    def break_into_tasks(self, feature_description, engine=None, temperature=0.7):
        try:
            prompt = TASK_BREAKER_USER.format(feature_description=feature_description)
            response = self.router.generate(
                prompt=prompt,
                system_prompt=TASK_BREAKER_SYSTEM,
                engine=engine,
                temperature=temperature
            )
            return self._parse_tasks(response)
        except Exception as e:
            return [{'number': 1, 'description': 'ERROR: ' + str(e), 'status': 'pending', 'code': '', 'explanation': '', 'is_error': True}]

    def _parse_tasks(self, response):
        if not response or not str(response).strip():
            return [{'number': 1, 'description': 'ERROR: Empty response received from model.', 'status': 'pending', 'code': '', 'explanation': '', 'is_error': True}]
        
        resp_str = str(response).strip()
        if resp_str.startswith('ERROR:'):
            return [{'number': 1, 'description': resp_str, 'status': 'pending', 'code': '', 'explanation': '', 'is_error': True}]
            
        tasks = []
        lines = resp_str.split('\n')
        task_num = 1
        for line in lines:
            line = line.strip()
            if not line:
                continue
            # Match 1., 1), -, *, • list formats
            match = re.match(r'^(?:(?:[0-9]+[.)\:-]+)|(?:[\-\*\•]+))\s*(.+)$', line)
            if match:
                desc = match.group(1).strip()
                desc = re.sub(r'^(?:Step\s*[0-9]+[\:\-\s]*)+', '', desc, flags=re.IGNORECASE).strip()
                if desc:
                    tasks.append({
                        'number': task_num,
                        'description': desc,
                        'status': 'pending',
                        'code': '',
                        'explanation': ''
                    })
                    task_num += 1
        
        # Fallback if model output wasn't formatted as a numbered list
        if not tasks:
            tasks.append({
                'number': 1,
                'description': resp_str if len(resp_str) < 300 else resp_str[:300] + '...',
                'status': 'pending',
                'code': '',
                'explanation': ''
            })
            
        return tasks

task_breaker = TaskBreaker(router)
print('TaskBreaker initialized')


TaskBreaker initialized


## 9. Code Generator (plan section 5.4 — `code_generator.py`)

For each subtask, generates code + explanation.

In [10]:
class CodeGenerator:
    """Generates code per step (plan section 5.4)."""

    def __init__(self, router):
        self.router = router

    def generate_code(self, task_description, project_description='', engine=None, temperature=0.4):
        prompt = CODE_GENERATOR_USER.format(
            task_description=task_description,
            project_description=project_description or 'General software project'
        )
        response = self.router.generate(
            prompt=prompt,
            system_prompt=CODE_GENERATOR_SYSTEM,
            engine=engine,
            temperature=temperature,
            max_tokens=4096
        )
        return self._parse_code_response(response)

    def explain_code(self, code, engine=None):
        prompt = CODE_EXPLAIN_USER.format(code=code)
        return self.router.generate(prompt=prompt, system_prompt=CODE_EXPLAIN_SYSTEM, engine=engine)

    def debug_code(self, code, error_description, engine=None):
        prompt = DEBUG_USER.format(code=code, error_description=error_description)
        return self.router.generate(prompt=prompt, system_prompt=DEBUG_SYSTEM, engine=engine)

    def _parse_code_response(self, response):
        code_pattern = r'```(?:\w+)?\n(.*?)```'
        matches = re.findall(code_pattern, response, re.DOTALL)
        if matches:
            code = '\n\n'.join(matches)
            explanation = re.sub(code_pattern, '', response, flags=re.DOTALL).strip()
        else:
            code = response
            explanation = ''
        return {'code': code, 'explanation': explanation, 'raw': response}

code_generator = CodeGenerator(router)
print('CodeGenerator initialized')

CodeGenerator initialized


## 10. SQLite History Database (plan section 5.4 — `history_db.py`)

Saves each plan/session to SQLite so past plans can be reopened.

In [19]:
import sqlite3
from datetime import datetime
from pathlib import Path

class HistoryDB:
    """SQLite read/write for saved plans (plan section 5.4)."""

    def __init__(self, db_path='data/history.db'):
        self.db_path = db_path
        Path(db_path).parent.mkdir(parents=True, exist_ok=True)
        self._init_db()

    def _init_db(self):
        with sqlite3.connect(self.db_path) as conn:
            conn.execute('''CREATE TABLE IF NOT EXISTS plans (
                id INTEGER PRIMARY KEY AUTOINCREMENT,
                title TEXT NOT NULL,
                description TEXT NOT NULL,
                engine TEXT DEFAULT 'local',
                created_at TEXT NOT NULL,
                updated_at TEXT NOT NULL
            )''')
            conn.execute('''CREATE TABLE IF NOT EXISTS tasks (
                id INTEGER PRIMARY KEY AUTOINCREMENT,
                plan_id INTEGER NOT NULL,
                task_order INTEGER NOT NULL,
                description TEXT NOT NULL,
                code TEXT DEFAULT '',
                explanation TEXT DEFAULT '',
                status TEXT DEFAULT 'pending',
                FOREIGN KEY (plan_id) REFERENCES plans(id) ON DELETE CASCADE
            )''')
            conn.commit()

    def save_plan(self, title, description, tasks, engine='local'):
        now = datetime.now().isoformat()
        with sqlite3.connect(self.db_path) as conn:
            cursor = conn.execute(
                'INSERT INTO plans (title, description, engine, created_at, updated_at) VALUES (?, ?, ?, ?, ?)',
                (title, description, engine, now, now)
            )
            plan_id = cursor.lastrowid
            for i, task in enumerate(tasks):
                conn.execute(
                    'INSERT INTO tasks (plan_id, task_order, description, code, explanation, status) VALUES (?, ?, ?, ?, ?, ?)',
                    (plan_id, i + 1, task['description'], task.get('code', ''), task.get('explanation', ''), task.get('status', 'pending'))
                )
            conn.commit()
        return plan_id

    def update_task(self, plan_id, task_order, code='', explanation='', status='done'):
        now = datetime.now().isoformat()
        with sqlite3.connect(self.db_path) as conn:
            conn.execute(
                'UPDATE tasks SET code=?, explanation=?, status=? WHERE plan_id=? AND task_order=?',
                (code, explanation, status, plan_id, task_order)
            )
            conn.execute('UPDATE plans SET updated_at=? WHERE id=?', (now, plan_id))
            conn.commit()

    def load_plan(self, plan_id):
        with sqlite3.connect(self.db_path) as conn:
            conn.row_factory = sqlite3.Row
            plan = conn.execute('SELECT * FROM plans WHERE id=?', (plan_id,)).fetchone()
            if not plan:
                return None
            tasks = conn.execute(
                'SELECT * FROM tasks WHERE plan_id=? ORDER BY task_order', (plan_id,)
            ).fetchall()
            return {
                'id': plan['id'], 'title': plan['title'],
                'description': plan['description'], 'engine': plan['engine'],
                'created_at': plan['created_at'], 'updated_at': plan['updated_at'],
                'tasks': [{
                    'number': t['task_order'], 'description': t['description'],
                    'code': t['code'], 'explanation': t['explanation'], 'status': t['status']
                } for t in tasks]
            }

    def list_plans(self):
        with sqlite3.connect(self.db_path) as conn:
            conn.row_factory = sqlite3.Row
            rows = conn.execute(
                'SELECT id, title, description, engine, created_at, updated_at FROM plans ORDER BY updated_at DESC'
            ).fetchall()
            return [dict(r) for r in rows]

    def delete_plan(self, plan_id):
        with sqlite3.connect(self.db_path) as conn:
            conn.execute('DELETE FROM tasks WHERE plan_id=?', (plan_id,))
            conn.execute('DELETE FROM plans WHERE id=?', (plan_id,))
            conn.commit()

history_db = HistoryDB()
print('HistoryDB initialized at ' + history_db.db_path)
print('  Existing plans: ' + str(len(history_db.list_plans())))

HistoryDB initialized at data/history.db
  Existing plans: 2


## 11. Interactive Planner UI

### 11a. Style and Header

In [12]:
import ipywidgets as widgets
from IPython.display import display, HTML, clear_output, FileLink
import html as html_module

# STATE
current_plan = {
    'id': None,
    'description': '',
    'tasks': [],
    'engine': DEFAULT_ENGINE
}

# STYLE
css_text = '<style>'
css_text += '.assistant-header { background: linear-gradient(135deg, #667eea 0%, #764ba2 100%); color: white; padding: 20px; border-radius: 12px; margin-bottom: 20px; font-family: sans-serif; }'
css_text += '.assistant-header h1 { margin: 0; font-size: 24px; }'
css_text += '.assistant-header p { margin: 5px 0 0 0; opacity: 0.9; font-size: 14px; }'
css_text += '.task-item { background: #f8f9fa; border: 1px solid #e9ecef; border-radius: 8px; padding: 12px 16px; margin: 6px 0; font-family: sans-serif; }'
css_text += '.task-done { border-left: 4px solid #28a745; }'
css_text += '.task-pending { border-left: 4px solid #ffc107; }'
css_text += '.code-output { background: #1e1e1e; color: #d4d4d4; padding: 16px; border-radius: 8px; font-family: monospace; font-size: 13px; overflow-x: auto; white-space: pre-wrap; margin: 8px 0; }'
css_text += '.status-bar { background: #e8f5e9; border: 1px solid #c8e6c9; border-radius: 8px; padding: 10px 16px; margin: 8px 0; font-family: sans-serif; color: #2e7d32; }'
css_text += '.error-bar { background: #ffebee; border: 1px solid #ffcdd2; border-radius: 8px; padding: 10px 16px; margin: 8px 0; font-family: sans-serif; color: #c62828; }'
css_text += '</style>'
display(HTML(css_text))

# HEADER
header_html = '<div class="assistant-header">'
header_html += '<h1>' + APP_TITLE + '</h1>'
header_html += '<p>Powered by Qwen 2.5 (local) + Optional Gemini API</p>'
header_html += '</div>'
display(HTML(header_html))

print('Style and header loaded.')

Style and header loaded.


### 11b. Widgets

In [13]:
# SETTINGS PANEL
engine_dropdown = widgets.Dropdown(
    options=['local', 'gemini'],
    value=DEFAULT_ENGINE,
    description='Engine:',
    style={'description_width': '80px'}
)

temperature_slider = widgets.FloatSlider(
    value=0.7, min=0.0, max=1.0, step=0.1,
    description='Temp:',
    style={'description_width': '80px'}
)

max_tokens_slider = widgets.IntSlider(
    value=2048, min=256, max=8192, step=256,
    description='Max Tokens:',
    style={'description_width': '80px'}
)

settings_box = widgets.VBox([
    widgets.HTML('<b>Settings</b>'),
    engine_dropdown,
    temperature_slider,
    max_tokens_slider
], layout=widgets.Layout(padding='10px', border='1px solid #ddd', border_radius='8px', margin='0 0 15px 0'))

display(settings_box)

# FEATURE INPUT
feature_input = widgets.Textarea(
    placeholder='Describe what you want to build...\nExample: REST API for a todo app with auth',
    layout=widgets.Layout(width='100%', height='100px')
)

generate_plan_btn = widgets.Button(
    description='Generate Plan',
    button_style='primary',
    layout=widgets.Layout(width='200px', height='40px')
)

plan_output = widgets.Output()
code_output = widgets.Output()
export_output = widgets.Output()
history_output = widgets.Output()

export_btn = widgets.Button(
    description='Export Plan as .md',
    button_style='info',
    layout=widgets.Layout(width='200px', height='35px')
)

history_dropdown = widgets.Dropdown(
    options=[],
    description='History:',
    style={'description_width': '80px'},
    layout=widgets.Layout(width='400px')
)

load_history_btn = widgets.Button(description='Load', button_style='success', layout=widgets.Layout(width='80px'))
delete_history_btn = widgets.Button(description='Delete', button_style='danger', layout=widgets.Layout(width='80px'))

# DISPLAY ALL WIDGETS
display(widgets.HTML('<b>Describe your feature:</b>'))
display(feature_input)
display(generate_plan_btn)
display(plan_output)
display(widgets.HTML('<hr><b>Generated Code</b>'))
display(code_output)
display(widgets.HTML('<hr>'))
display(export_btn)
display(export_output)
display(widgets.HTML('<hr><b>Plan History</b>'))
display(widgets.HBox([history_dropdown, load_history_btn, delete_history_btn]))
display(history_output)

print('Widgets created.')

HTML(value='<b>Describe your feature:</b>')

Textarea(value='', layout=Layout(height='100px', width='100%'), placeholder='Describe what you want to build..…

Button(button_style='primary', description='Generate Plan', layout=Layout(height='40px', width='200px'), style…

Output()

HTML(value='<hr><b>Generated Code</b>')

...

Button(button_style='info', description='Export Plan as .md', layout=Layout(height='35px', width='200px'), sty…

Output()

HTML(value='<hr><b>Plan History</b>')

Output()

Widgets created.


### 11c. Helper Functions

In [25]:
def refresh_history_dropdown():
    """Refresh the history dropdown with saved plans."""
    plans = history_db.list_plans()
    if plans:
        history_dropdown.options = [
            (p['title'] + ' (' + p['created_at'][:10] + ')', p['id']) for p in plans
        ]
    else:
        history_dropdown.options = [('No saved plans', -1)]


def render_tasks(tasks):
    """Render task list with generate-code buttons."""
    with plan_output:
        clear_output()
        if not tasks:
            display(HTML('<div class="error-bar">No tasks generated. Try again.</div>'))
            return
        display(HTML('<div class="status-bar">Generated ' + str(len(tasks)) + ' tasks</div>'))
        for i, task in enumerate(tasks):
            sc = 'task-done' if task['status'] == 'done' else 'task-pending'
            si = '[DONE]' if task['status'] == 'done' else '[TODO]'
            desc = html_module.escape(task['description'])
            task_html = '<div class="task-item ' + sc + '">' + si + ' <b>' + str(task['number']) + '.</b> ' + desc + '</div>'
            display(HTML(task_html))
            gen_btn = widgets.Button(
                description='Generate Code for Task ' + str(task['number']),
                button_style='warning' if task['status'] != 'done' else 'success',
                layout=widgets.Layout(width='300px', margin='0 0 10px 0')
            )
            def make_handler(idx):
                def handler(b):
                    on_generate_code(idx)
                return handler
            gen_btn.on_click(make_handler(i))
            display(gen_btn)


def render_code(task_num, result):
    """Render generated code in the code output area."""
    with code_output:
        display(HTML('<h4>Task ' + str(task_num) + ' - Code</h4>'))
        escaped = html_module.escape(result['code'])
        display(HTML('<div class="code-output">' + escaped + '</div>'))
        if result.get('explanation'):
            exp = html_module.escape(result['explanation'])
            display(HTML('<p><b>Explanation:</b> ' + exp + '</p>'))
        display(HTML('<hr>'))


print('Helper functions defined.')

Helper functions defined.


### 11d. Event Handlers & Wiring

In [26]:
def on_generate_plan(btn):
    """Handle Generate Plan button click."""
    description = feature_input.value.strip()
    if not description:
        with plan_output:
            clear_output()
            display(HTML('<div class="error-bar">Please enter a feature description first.</div>'))
        return
    with plan_output:
        clear_output()
        display(HTML('<div class="status-bar">Generating plan... (this may take a moment)</div>'))
    with code_output:
        clear_output()
    engine = engine_dropdown.value
    tasks = task_breaker.break_into_tasks(description, engine=engine, temperature=temperature_slider.value)
    current_plan['description'] = description
    current_plan['tasks'] = tasks
    current_plan['engine'] = engine
    title = description[:50]
    if len(description) > 50:
        title += '...'
    plan_id = history_db.save_plan(title, description, tasks, engine)
    current_plan['id'] = plan_id
    render_tasks(tasks)
    refresh_history_dropdown()


def on_generate_code(task_idx):
    """Handle Generate Code button click for a specific task."""
    task = current_plan['tasks'][task_idx]
    with code_output:
        display(HTML('<div class="status-bar">Generating code for Task ' + str(task['number']) + '...</div>'))
    result = code_generator.generate_code(
        task_description=task['description'],
        project_description=current_plan['description'],
        engine=engine_dropdown.value,
        temperature=temperature_slider.value
    )
    task['code'] = result['code']
    task['explanation'] = result['explanation']
    task['status'] = 'done'
    if current_plan['id']:
        history_db.update_task(
            current_plan['id'], task['number'],
            code=result['code'], explanation=result['explanation'], status='done'
        )
    render_code(task['number'], result)
    render_tasks(current_plan['tasks'])


def on_export(btn):
    """Export full plan + code as .md file."""
    with export_output:
        clear_output()
        if not current_plan['tasks']:
            display(HTML('<div class="error-bar">No plan to export. Generate a plan first.</div>'))
            return
        md_lines = []
        md_lines.append('# ' + current_plan['description'] + '\n\n')
        md_lines.append('**Engine:** ' + current_plan['engine'] + '\n\n')
        md_lines.append('---\n\n## Tasks\n\n')
        for task in current_plan['tasks']:
            si = '[DONE]' if task['status'] == 'done' else '[TODO]'
            md_lines.append(si + ' **' + str(task['number']) + '.** ' + task['description'] + '\n\n')
            if task.get('code'):
                md_lines.append('```\n' + task['code'] + '\n```\n\n')
            if task.get('explanation'):
                md_lines.append('*' + task['explanation'] + '*\n\n')
            md_lines.append('---\n\n')
        md_content = ''.join(md_lines)
        export_path = 'exported_plan.md'
        with open(export_path, 'w') as f:
            f.write(md_content)
        display(HTML('<div class="status-bar">Plan exported to <b>' + export_path + '</b></div>'))
        display(FileLink(export_path, result_html_prefix='Download: '))


def on_load_history(btn):
    """Load a saved plan from history."""
    plan_id = history_dropdown.value
    if plan_id == -1:
        return
    plan_data = history_db.load_plan(plan_id)
    if plan_data:
        current_plan['id'] = plan_data['id']
        current_plan['description'] = plan_data['description']
        current_plan['tasks'] = plan_data['tasks']
        current_plan['engine'] = plan_data['engine']
        feature_input.value = plan_data['description']
        engine_dropdown.value = plan_data['engine']
        render_tasks(plan_data['tasks'])
        with code_output:
            clear_output()
            for task in plan_data['tasks']:
                if task.get('code'):
                    render_code(task['number'], {'code': task['code'], 'explanation': task.get('explanation', '')})
        with history_output:
            clear_output()
            display(HTML('<div class="status-bar">Loaded plan: ' + plan_data['title'] + '</div>'))


def on_delete_history(btn):
    """Delete a saved plan from history."""
    plan_id = history_dropdown.value
    if plan_id == -1:
        return
    history_db.delete_plan(plan_id)
    refresh_history_dropdown()
    with history_output:
        clear_output()
        display(HTML('<div class="status-bar">Plan deleted.</div>'))


# WIRE UP EVENTS
generate_plan_btn.on_click(on_generate_plan)
export_btn.on_click(on_export)
load_history_btn.on_click(on_load_history)
delete_history_btn.on_click(on_delete_history)

# Initial history load
refresh_history_dropdown()

print('UI ready! Describe a feature above and click Generate Plan.')

UI ready! Describe a feature above and click Generate Plan.


---

## 12. Quick Test (Optional)

Run the cell below to test the core components directly without the UI.

In [28]:
# --- Quick Test ---
# Uncomment and run to test individual components:

# Test 1: Direct Qwen chat
# response = qwen_client.generate('Write a Python hello world function', system_prompt='You are a helpful coding assistant.')
# print(response)

# Test 2: Task breaker
# tasks = task_breaker.break_into_tasks('Build a REST API for a todo app with authentication')
# for t in tasks:
#     print(str(t['number']) + '. ' + t['description'])

# Test 3: Code generator
# result = code_generator.generate_code('Create a Python Flask app with a /health endpoint')
# print(result['code'])

# Test 4: History DB round-trip
# test_tasks = [{'number': 1, 'description': 'Test task', 'status': 'pending', 'code': '', 'explanation': ''}]
# pid = history_db.save_plan('Test Plan', 'Testing history', test_tasks)
# loaded = history_db.load_plan(pid)
# print('Saved and loaded plan: ' + loaded['title'])
# history_db.delete_plan(pid)
# print('Cleaned up test plan')

# print('Uncomment the tests above to run them.')

---

## Notes

- Keep `.env` out of version control (`.gitignore` should include it).
- Qwen 2.5 runs fully offline once pulled — good for privacy-sensitive code.
- Gemini is optional and only activates when a key is present and selected.
- All plans are saved to `data/history.db` automatically.